# Adversarial Attack Visualization for TAMPAR

This notebook generates visualizations comparing:
1. **Unattacked** UV map surface with its change detection heatmap
2. **FGSM/PGD attacked** UV map surface with its change detection heatmap
3. **Perturbation heatmap** showing the adversarial noise

Output format similar to the C&W / PGD comparison figure.

In [ ]:
# Install dependencies (run once in Colab)
# !pip install torch torchvision matplotlib numpy opencv-python pillow

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import time

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
# ==================== CONFIGURE THESE PATHS ====================

# Path to the reference UV map (original untampered parcel)
REFERENCE_UVMAP_PATH = "data/tampar_sample/uvmaps/id_01_uvmap.png"

# Path to the detected UV map (potentially tampered parcel observation)
DETECTED_UVMAP_PATH = "data/tampar_sample/validation/id_01_20230516_142710_uvmap_pred.png"

# Which surface to extract (0-8, where 1=top, 3=left, 4=center, 5=right, 7=bottom)
# Based on 3x3 grid: [0,1,2; 3,4,5; 6,7,8] -> only 1,3,4,5,7 are actual surfaces
SURFACE_INDEX = 4  # center surface
SURFACE_NAME = "center"

# Attack parameters
EPSILON_FGSM = 0.03  # FGSM perturbation magnitude (L-inf)
EPSILON_PGD = 0.03   # PGD perturbation magnitude (L-inf)
PGD_STEPS = 40       # Number of PGD iterations
PGD_ALPHA = 0.01     # PGD step size

# Output settings
SAVE_OUTPUT = True
OUTPUT_PATH = "adversarial_comparison.png"

## Helper Functions

In [ ]:
def load_uvmap(path):
    """Load UV map image and convert to RGB."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def get_surface_patches(uvmap, grid_size=3):
    """
    Extract surface patches from UV map.
    Returns list of 9 patches (3x3 grid).
    Indices: [0,1,2; 3,4,5; 6,7,8]
    Surfaces: 1=top, 3=left, 4=center, 5=right, 7=bottom
    """
    h, w = uvmap.shape[:2]
    patch_h, patch_w = h // grid_size, w // grid_size
    
    patches = []
    for i in range(grid_size):
        for j in range(grid_size):
            patch = uvmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w]
            patches.append(patch)
    return patches


def numpy_to_tensor(img):
    """Convert numpy image (H,W,C) to tensor (1,C,H,W) normalized to [0,1]."""
    img = img.astype(np.float32) / 255.0
    tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device)


def tensor_to_numpy(tensor):
    """Convert tensor (1,C,H,W) to numpy image (H,W,C) in [0,255]."""
    img = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img = np.clip(img * 255, 0, 255).astype(np.uint8)
    return img

## Simple Similarity Model (Surrogate for Attacks)

Since we need gradients for adversarial attacks, we create a differentiable similarity model.

In [ ]:
class DifferentiableSimilarity(nn.Module):
    """
    A differentiable similarity measure between two images.
    Used as a surrogate model for adversarial attacks.
    
    The goal of the attacker is to make tampered images look similar
    to the reference (increase similarity score), fooling the detector.
    """
    def __init__(self):
        super().__init__()
        # Simple CNN feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        
    def forward(self, img1, img2):
        """Compute similarity score between two images."""
        feat1 = self.features(img1).flatten(1)
        feat2 = self.features(img2).flatten(1)
        # Cosine similarity
        similarity = F.cosine_similarity(feat1, feat2)
        return similarity


# Initialize model
similarity_model = DifferentiableSimilarity().to(device)
similarity_model.eval()
print("Similarity model initialized")

## Adversarial Attack Functions

In [ ]:
def fgsm_attack(reference_tensor, detected_tensor, model, epsilon):
    """
    Fast Gradient Sign Method (FGSM) attack.
    
    Goal: Perturb the detected image to increase similarity with reference,
    making tampering harder to detect.
    
    Args:
        reference_tensor: Reference (untampered) image tensor
        detected_tensor: Detected (potentially tampered) image tensor
        model: Differentiable similarity model
        epsilon: Perturbation magnitude
    
    Returns:
        adversarial_tensor: Perturbed image tensor
        perturbation: The adversarial perturbation
    """
    start_time = time.time()
    
    detected_tensor = detected_tensor.clone().detach().requires_grad_(True)
    
    # Compute similarity (we want to maximize this to fool detector)
    similarity = model(reference_tensor, detected_tensor)
    
    # Backward pass
    model.zero_grad()
    similarity.backward()
    
    # FGSM: move in direction of gradient to increase similarity
    perturbation = epsilon * detected_tensor.grad.sign()
    adversarial_tensor = detected_tensor + perturbation
    adversarial_tensor = torch.clamp(adversarial_tensor, 0, 1)
    
    elapsed_time = time.time() - start_time
    
    # Compute L2 norm of perturbation
    l2_norm = torch.norm(perturbation).item()
    
    return adversarial_tensor.detach(), perturbation.detach(), l2_norm, elapsed_time


def pgd_attack(reference_tensor, detected_tensor, model, epsilon, alpha, num_steps):
    """
    Projected Gradient Descent (PGD) attack.
    
    Iterative version of FGSM with projection back to epsilon-ball.
    
    Args:
        reference_tensor: Reference (untampered) image tensor
        detected_tensor: Detected (potentially tampered) image tensor
        model: Differentiable similarity model
        epsilon: Maximum perturbation magnitude (L-inf)
        alpha: Step size for each iteration
        num_steps: Number of iterations
    
    Returns:
        adversarial_tensor: Perturbed image tensor
        perturbation: The adversarial perturbation
    """
    start_time = time.time()
    
    original_tensor = detected_tensor.clone().detach()
    adversarial_tensor = detected_tensor.clone().detach()
    
    for step in range(num_steps):
        adversarial_tensor.requires_grad_(True)
        
        # Compute similarity
        similarity = model(reference_tensor, adversarial_tensor)
        
        # Backward pass
        model.zero_grad()
        similarity.backward()
        
        # PGD step: move in direction of gradient
        with torch.no_grad():
            adversarial_tensor = adversarial_tensor + alpha * adversarial_tensor.grad.sign()
            
            # Project back to epsilon-ball around original
            perturbation = adversarial_tensor - original_tensor
            perturbation = torch.clamp(perturbation, -epsilon, epsilon)
            adversarial_tensor = original_tensor + perturbation
            adversarial_tensor = torch.clamp(adversarial_tensor, 0, 1)
    
    elapsed_time = time.time() - start_time
    
    # Final perturbation
    perturbation = adversarial_tensor - original_tensor
    l2_norm = torch.norm(perturbation).item()
    
    return adversarial_tensor.detach(), perturbation.detach(), l2_norm, elapsed_time

## Change Detection Heatmap (Simplified SimSaC-like)

In [ ]:
def compute_change_heatmap(reference, detected):
    """
    Compute a change detection heatmap between reference and detected images.
    
    This is a simplified version - in practice you'd use SimSaC.
    Uses multi-scale difference analysis.
    
    Args:
        reference: Reference image (H,W,C) numpy array
        detected: Detected image (H,W,C) numpy array
    
    Returns:
        heatmap: Change detection heatmap (H,W) numpy array
    """
    # Convert to grayscale
    ref_gray = cv2.cvtColor(reference, cv2.COLOR_RGB2GRAY).astype(np.float32)
    det_gray = cv2.cvtColor(detected, cv2.COLOR_RGB2GRAY).astype(np.float32)
    
    # Multi-scale difference
    heatmap = np.zeros_like(ref_gray)
    
    for scale in [1, 2, 4]:
        if scale > 1:
            ref_scaled = cv2.resize(ref_gray, None, fx=1/scale, fy=1/scale)
            det_scaled = cv2.resize(det_gray, None, fx=1/scale, fy=1/scale)
        else:
            ref_scaled = ref_gray
            det_scaled = det_gray
        
        # Compute absolute difference
        diff = np.abs(ref_scaled - det_scaled)
        
        # Apply Gaussian blur to smooth
        diff = cv2.GaussianBlur(diff, (5, 5), 0)
        
        # Resize back to original size
        if scale > 1:
            diff = cv2.resize(diff, (ref_gray.shape[1], ref_gray.shape[0]))
        
        heatmap += diff / scale
    
    # Normalize to [0, 255]
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8) * 255
    
    return heatmap.astype(np.uint8)


def compute_perturbation_heatmap(perturbation_tensor):
    """
    Visualize the adversarial perturbation as a heatmap.
    
    Args:
        perturbation_tensor: Perturbation tensor (1,C,H,W)
    
    Returns:
        heatmap: Perturbation magnitude heatmap (H,W) numpy array
    """
    # Get magnitude across channels
    perturbation = perturbation_tensor.squeeze(0).cpu().numpy()
    magnitude = np.sqrt(np.sum(perturbation ** 2, axis=0))
    
    # Normalize to [0, 255]
    magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8) * 255
    
    return magnitude.astype(np.uint8)

## Load Images and Extract Surface Patch

In [ ]:
# Load UV maps
print(f"Loading reference UV map: {REFERENCE_UVMAP_PATH}")
reference_uvmap = load_uvmap(REFERENCE_UVMAP_PATH)
print(f"Reference shape: {reference_uvmap.shape}")

print(f"\nLoading detected UV map: {DETECTED_UVMAP_PATH}")
detected_uvmap = load_uvmap(DETECTED_UVMAP_PATH)
print(f"Detected shape: {detected_uvmap.shape}")

# Extract surface patches
reference_patches = get_surface_patches(reference_uvmap)
detected_patches = get_surface_patches(detected_uvmap)

# Get the selected surface
reference_surface = reference_patches[SURFACE_INDEX]
detected_surface = detected_patches[SURFACE_INDEX]

print(f"\nExtracted {SURFACE_NAME} surface (index {SURFACE_INDEX})")
print(f"Surface patch shape: {reference_surface.shape}")

# Display the surfaces
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(reference_surface)
axes[0].set_title(f'Reference {SURFACE_NAME} surface', fontsize=12)
axes[0].axis('off')

axes[1].imshow(detected_surface)
axes[1].set_title(f'Detected {SURFACE_NAME} surface', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Generate Adversarial Examples

In [ ]:
# Convert to tensors
reference_tensor = numpy_to_tensor(reference_surface)
detected_tensor = numpy_to_tensor(detected_surface)

print("Generating adversarial examples...")

# FGSM Attack
print(f"\n1. FGSM Attack (epsilon={EPSILON_FGSM})")
fgsm_adv_tensor, fgsm_perturbation, fgsm_l2, fgsm_time = fgsm_attack(
    reference_tensor, detected_tensor, similarity_model, EPSILON_FGSM
)
fgsm_adv_image = tensor_to_numpy(fgsm_adv_tensor)
print(f"   L2 norm: {fgsm_l2:.6f}")
print(f"   Time: {fgsm_time:.2f}s")

# PGD Attack
print(f"\n2. PGD Attack (epsilon={EPSILON_PGD}, steps={PGD_STEPS})")
pgd_adv_tensor, pgd_perturbation, pgd_l2, pgd_time = pgd_attack(
    reference_tensor, detected_tensor, similarity_model, EPSILON_PGD, PGD_ALPHA, PGD_STEPS
)
pgd_adv_image = tensor_to_numpy(pgd_adv_tensor)
print(f"   L2 norm: {pgd_l2:.6f}")
print(f"   Time: {pgd_time:.2f}s")

## Compute Change Detection Heatmaps

In [ ]:
print("Computing change detection heatmaps...")

# Original (unattacked) change heatmap
original_heatmap = compute_change_heatmap(reference_surface, detected_surface)

# FGSM attacked change heatmap
fgsm_heatmap = compute_change_heatmap(reference_surface, fgsm_adv_image)

# PGD attacked change heatmap
pgd_heatmap = compute_change_heatmap(reference_surface, pgd_adv_image)

# Perturbation heatmaps
fgsm_perturbation_heatmap = compute_perturbation_heatmap(fgsm_perturbation)
pgd_perturbation_heatmap = compute_perturbation_heatmap(pgd_perturbation)

print("Done!")

## Create Final Visualization

In [ ]:
# Custom colormap for heatmaps (similar to the reference image)
colors = ['black', 'darkred', 'red', 'orange', 'yellow', 'white']
n_bins = 256
cmap_hot = LinearSegmentedColormap.from_list('custom_hot', colors, N=n_bins)


def create_comparison_figure():
    """
    Create a figure comparing original vs adversarial examples.
    Layout:
    Row 1: Original | FGSM Adversarial | Perturbation Heatmap
    Row 2: Original | PGD Adversarial  | Perturbation Heatmap
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Row 1: FGSM
    axes[0, 0].imshow(detected_surface)
    axes[0, 0].set_title('Original\n(Unattacked)', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(fgsm_adv_image)
    axes[0, 1].set_title(f'FGSM Attack\n(ε={EPSILON_FGSM})', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    im1 = axes[0, 2].imshow(fgsm_perturbation_heatmap, cmap=cmap_hot)
    axes[0, 2].set_title(f'Perturbation Heatmap\nL2: {fgsm_l2:.4f}, Time: {fgsm_time:.2f}s', 
                         fontsize=14, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Row 2: PGD
    axes[1, 0].imshow(detected_surface)
    axes[1, 0].set_title('Original\n(Unattacked)', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(pgd_adv_image)
    axes[1, 1].set_title(f'PGD Attack\n(ε={EPSILON_PGD}, steps={PGD_STEPS})', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    im2 = axes[1, 2].imshow(pgd_perturbation_heatmap, cmap=cmap_hot)
    axes[1, 2].set_title(f'Perturbation Heatmap\nL2: {pgd_l2:.4f}, Time: {pgd_time:.2f}s', 
                         fontsize=14, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.suptitle(f'Adversarial Attack Comparison - {SURFACE_NAME.upper()} Surface', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    if SAVE_OUTPUT:
        plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"\nSaved figure to: {OUTPUT_PATH}")
    
    plt.show()


create_comparison_figure()

## Alternative Layout: With Change Detection Heatmaps

In [ ]:
def create_full_comparison_figure():
    """
    Create a comprehensive figure showing:
    - Original image and its change heatmap
    - FGSM adversarial and its change heatmap + perturbation
    - PGD adversarial and its change heatmap + perturbation
    """
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    
    # Row 1: Original
    axes[0, 0].imshow(detected_surface)
    axes[0, 0].set_title('Original Surface\n(Detected UV Map)', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(original_heatmap, cmap=cmap_hot)
    axes[0, 1].set_title('Change Detection Heatmap\n(vs Reference)', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Empty cell for original (no perturbation)
    axes[0, 2].text(0.5, 0.5, 'No Attack\n(Baseline)', ha='center', va='center', 
                    fontsize=14, transform=axes[0, 2].transAxes)
    axes[0, 2].axis('off')
    
    # Row 2: FGSM
    axes[1, 0].imshow(fgsm_adv_image)
    axes[1, 0].set_title(f'FGSM Adversarial\n(ε={EPSILON_FGSM})', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(fgsm_heatmap, cmap=cmap_hot)
    axes[1, 1].set_title('Change Detection Heatmap\n(After FGSM Attack)', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(fgsm_perturbation_heatmap, cmap=cmap_hot)
    axes[1, 2].set_title(f'FGSM Perturbation\nL2: {fgsm_l2:.4f}, Time: {fgsm_time:.2f}s', 
                         fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    # Row 3: PGD
    axes[2, 0].imshow(pgd_adv_image)
    axes[2, 0].set_title(f'PGD Adversarial\n(ε={EPSILON_PGD}, steps={PGD_STEPS})', fontsize=12, fontweight='bold')
    axes[2, 0].axis('off')
    
    axes[2, 1].imshow(pgd_heatmap, cmap=cmap_hot)
    axes[2, 1].set_title('Change Detection Heatmap\n(After PGD Attack)', fontsize=12, fontweight='bold')
    axes[2, 1].axis('off')
    
    axes[2, 2].imshow(pgd_perturbation_heatmap, cmap=cmap_hot)
    axes[2, 2].set_title(f'PGD Perturbation\nL2: {pgd_l2:.4f}, Time: {pgd_time:.2f}s', 
                         fontsize=12, fontweight='bold')
    axes[2, 2].axis('off')
    
    # Add row labels
    fig.text(0.02, 0.83, 'Baseline', fontsize=14, fontweight='bold', rotation=90, va='center')
    fig.text(0.02, 0.5, 'FGSM', fontsize=14, fontweight='bold', rotation=90, va='center')
    fig.text(0.02, 0.17, 'PGD', fontsize=14, fontweight='bold', rotation=90, va='center')
    
    # Add column labels
    fig.text(0.22, 0.98, 'Surface Image', fontsize=14, fontweight='bold', ha='center')
    fig.text(0.52, 0.98, 'Change Heatmap', fontsize=14, fontweight='bold', ha='center')
    fig.text(0.82, 0.98, 'Perturbation', fontsize=14, fontweight='bold', ha='center')
    
    plt.suptitle(f'Adversarial Attack Analysis - {SURFACE_NAME.upper()} Surface\n', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    if SAVE_OUTPUT:
        full_output_path = OUTPUT_PATH.replace('.png', '_full.png')
        plt.savefig(full_output_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"\nSaved full figure to: {full_output_path}")
    
    plt.show()


create_full_comparison_figure()

## Summary Statistics

In [ ]:
print("="*60)
print("ADVERSARIAL ATTACK SUMMARY")
print("="*60)
print(f"\nSurface: {SURFACE_NAME.upper()}")
print(f"Reference: {REFERENCE_UVMAP_PATH}")
print(f"Detected: {DETECTED_UVMAP_PATH}")

print(f"\n{'Attack':<10} {'Epsilon':<10} {'L2 Norm':<12} {'Time (s)':<10}")
print("-"*45)
print(f"{'FGSM':<10} {EPSILON_FGSM:<10.3f} {fgsm_l2:<12.6f} {fgsm_time:<10.2f}")
print(f"{'PGD':<10} {EPSILON_PGD:<10.3f} {pgd_l2:<12.6f} {pgd_time:<10.2f}")

print(f"\nPGD Parameters: steps={PGD_STEPS}, alpha={PGD_ALPHA}")
print("\nNote: Lower L2 norm = less visible perturbation")
print("      PGD is more iterative but often more effective")

## Usage Instructions

1. **Update paths** in the Configuration cell:
   - `REFERENCE_UVMAP_PATH`: Path to the original (untampered) UV map
   - `DETECTED_UVMAP_PATH`: Path to the detected (potentially tampered) UV map

2. **Select surface** to analyze:
   - `SURFACE_INDEX`: 1=top, 3=left, 4=center, 5=right, 7=bottom

3. **Adjust attack parameters** if needed:
   - `EPSILON_FGSM/PGD`: Perturbation magnitude (smaller = less visible)
   - `PGD_STEPS`: More steps = stronger attack but slower

4. **Run all cells** to generate visualizations

5. **Output**: Two figures saved:
   - `adversarial_comparison.png`: Simple comparison
   - `adversarial_comparison_full.png`: Full analysis with heatmaps